# HJNC 자부두 환적 컨테이너 야드 체류시간 및 선박 운영시간 분석

**분석 대상:** `hjnc_자부두환적.csv`의 컨테이너 1,490건과 `HJNC_선석스케줄.csv`의 선박 운항 기록(1,002항차). 야드 체류시간은 **`온샤시 시간 − 야드 장치 시간`**으로 계산한다.

**분석 흐름:** 야드 체류시간 산출 → 데이터 검증 및 기초 통계 → 구간별 분포 → 적하모선항차와 선석스케줄 연결 → 현재 선박 및 직전 입항 선박의 시간 지표와 체류시간 관계 검토 → 분석 한계 및 향후 시뮬레이션 방향.

**해석 유의:** 관측 데이터의 상관관계는 인과관계를 의미하지 않으며, 입·출항 지연시간은 반입 일정 기준 경과시간을 이용한 **대체지표**이다.




## 1. 분석용 컨테이너 데이터 불러오기


In [17]:
import pandas as pd
from pathlib import Path


data_dirs = [Path.cwd(), Path(r"C:\project\data\hjnc_data"), Path(r"C:\project\analysis\TS")]
candidates = [folder / name for folder in data_dirs for name in (
    "hjnc_자부두환적.csv", "hjnc_자부두환적.csv"
)]
file_path = next((p for p in candidates if p.exists()), None)
if file_path is None:
    raise FileNotFoundError(
        "hjnc_자부두환적(1).csv 파일을 노트북과 같은 폴더 또는 "
        r"C:\project\data\hjnc_data 에 놓아주세요."
    )

df = pd.read_csv(file_path, encoding="utf-8-sig", low_memory=False)
df.columns = df.columns.str.strip()


df = df.rename(columns={"야드 장치시간": "야드 장치 시간"})

required_cols = ["컨테이너번호", "야드 장치 시간", "온샤시 시간", "적하모선항차"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise KeyError(f"분석에 필요한 CSV 컬럼이 없습니다: {missing}")

print("읽은 CSV:", file_path)
print("전체 컨테이너 기록:", len(df))
print("\n컬럼 목록:")
print(df.columns.tolist())
print("\n주요 시간 컬럼 결측치:")
time_cols_initial = ["반입시간", "반출시간", "야드 장치 시간", "온샤시 시간"]
display(df[time_cols_initial].isna().sum().to_frame("결측치"))
display(df.head())

읽은 CSV: C:\project\data\hjnc_data\hjnc_자부두환적.csv
전체 컨테이너 기록: 1490

컬럼 목록:
['컨테이너번호', '선사코드', '적/공', 'Size/Type', '화물종류', '중량(kg)', 'Seal No', 'BL No', '양하항', '적하항', '야드위치', '반입방식', '반입시간', '반출방식', '반출시간', '반입차량번호', '반출차량번호', '손상여부', 'IMDG', 'UNNO', '온도', '양하모선명', '양하모선항차', '적하모선명', '적하모선항차', '선내위치', '야드 장치 시간', '온샤시 시간', '경과료 Free Date', '냉동료 Free Date', '반출예정일']

주요 시간 컬럼 결측치:


,결측치
반입시간,0
반출시간,0
야드 장치 시간,0
온샤시 시간,0


,컨테이너번호,선사코드,적/공,Size/Type,화물종류,중량(kg),Seal No,BL No,양하항,적하항,...,양하모선명,양하모선항차,적하모선명,적하모선항차,선내위치,야드 장치 시간,온샤시 시간,경과료 Free Date,냉동료 Free Date,반출예정일
0,AMFU8667185,SML,F,40/GE(4500),G,26950.0,seal1:11994500,PHX5A5110700,KRKAN,KRPUS,...,SM PORTLAND,SMPL-006/2025,SM NINGBO,SMNB-007/2025,10-09-82,2025-05-11 18:12:55,2025-05-31 15:08:56,2025-05-22 23:59:59,NaN,NaN
1,AXIU1350376,SML,F,40/GE(4510),G,26318.0,seal1:0020158,PHX5A4854700,KRKAN,KRPUS,...,SM QINGDAO,SMQT-005/2025,SM YANTIAN,SMYT-005/2025,42-03-88,2025-05-02 01:52:19,2025-05-05 04:29:04,2025-05-13 23:59:59,NaN,NaN
2,AXIU4223734,SML,F,4210,G,22620.0,seal1:SML221951,SZP5J5662500,USPDX,KRPUS,...,SM SEOUL,SMSE-005/2025,SM SHANGHAI,SMSH-012/2025,22-02-12,2025-09-08 01:31:40,2025-09-12 12:57:30,2025-09-19 23:59:59,NaN,NaN
3,BEAU4507198,YML,F,40/GE(45G0),G,4580.0,seal1:UL1268624,YMJAE125365026,CNSHA,KRPUS,...,ONE CONTINUITY,OOCN-004/2025,HMM DUBLIN,HODU-002/2025,06-08-80,2025-09-27 16:00:29,2025-10-03 01:27:46,2025-10-12 23:59:59,NaN,NaN
4,BEAU4549233,HLC,F,40/GE(4500),G,13994.0,seal1:HLK0749815,HLCUNG12412VEIY0,CAVA5,KRPUS,...,TSINGTAO EXPRESS,TSXX-001/2025,YM TIPTOP,YIPT-001/2025,46-00-82,2025-02-11 17:27:03,2025-02-15 23:18:19,2025-02-22 23:59:59,NaN,NaN


## 2. 야드 체류시간 계산

In [18]:
# 시간 컬럼 변환
df["야드 장치 시간"] = pd.to_datetime(
    df["야드 장치 시간"], errors="coerce"
)

df["온샤시 시간"] = pd.to_datetime(
    df["온샤시 시간"], errors="coerce"
)

# 야드 체류시간 계산 (시간 단위)
df["야드체류시간(h)"] = (
    (df["온샤시 시간"] - df["야드 장치 시간"]).dt.total_seconds() / 3600
).round(1)

# 계산 결과 확인
print("전체 데이터:", len(df))
print("계산 가능한 데이터:", df["야드체류시간(h)"].notna().sum())
print("음수 체류시간:", (df["야드체류시간(h)"] < 0).sum())

print("\n야드 체류시간 통계 (시간)")
display(df["야드체류시간(h)"].describe())

print("\n상위 5개 데이터")
display(
    df[
        [
            "컨테이너번호",
            "야드 장치 시간",
            "온샤시 시간",
            "야드체류시간(h)"
        ]
    ].head()
)

전체 데이터: 1490
계산 가능한 데이터: 1490
음수 체류시간: 0

야드 체류시간 통계 (시간)


count    1490.000000
mean      195.897987
std       117.270839
min         1.900000
25%       114.150000
50%       168.900000
75%       261.975000
max       885.000000
Name: 야드체류시간(h), dtype: float64


상위 5개 데이터


,컨테이너번호,야드 장치 시간,온샤시 시간,야드체류시간(h)
0,AMFU8667185,2025-05-11 18:12:55,2025-05-31 15:08:56,476.9
1,AXIU1350376,2025-05-02 01:52:19,2025-05-05 04:29:04,74.6
2,AXIU4223734,2025-09-08 01:31:40,2025-09-12 12:57:30,107.4
3,BEAU4507198,2025-09-27 16:00:29,2025-10-03 01:27:46,129.5
4,BEAU4549233,2025-02-11 17:27:03,2025-02-15 23:18:19,101.9


## 3. 야드 체류시간 데이터 검증 및 기초 통계 분석

```
분석 목적
- 정상적으로 계산된 컨테이너가 몇 건인지 확인
- 음수 또는 0시간인 데이터 확인
- 평균·중앙값·최솟값·최댓값 비교
- 체류시간이 긴 컨테이너가 평균에 영향을 주는지 확인
```

In [19]:


col = "야드체류시간(h)"


print("전체 컨테이너 기록:", len(df))
print("체류시간 결측치:", df[col].isna().sum())
print("음수 체류시간:", (df[col] < 0).sum())
print("0시간 체류:", (df[col] == 0).sum())
print("정상 체류시간:", (df[col] > 0).sum())


print("\n========== 야드 체류시간 기초 통계 ==========")

summary = pd.DataFrame({
    "지표": [
        "평균",
        "중앙값",
        "표준편차",
        "최솟값",
        "25% 분위수",
        "75% 분위수",
        "최댓값"
    ],
    "시간(h)": [
        df[col].mean(),
        df[col].median(),
        df[col].std(),
        df[col].min(),
        df[col].quantile(0.25),
        df[col].quantile(0.75),
        df[col].max()
    ]
})

summary["시간(h)"] = summary["시간(h)"].round(1)

display(summary)

전체 컨테이너 기록: 1490
체류시간 결측치: 0
음수 체류시간: 0
0시간 체류: 0
정상 체류시간: 1490

========== 야드 체류시간 기초 통계 ==========


,지표,시간(h)
0,평균,195.9
1,중앙값,168.9
2,표준편차,117.3
3,최솟값,1.9
4,25% 분위수,114.2
5,75% 분위수,262.0
6,최댓값,885.0


### 1. 평균 및 중앙값 비교

평균 야드 체류시간은 **195.9시간**, 중앙값은 **168.9시간**으로 평균이 **27.0시간** 길었다. 일부 장기 체류가 평균을 높이는 방향으로 작용했을 가능성이 있어 중앙값도 함께 확인한다.

### 2. 야드 체류시간 분포 및 편차 분석

25% 분위수는 **114.2시간**, 75% 분위수는 **262.0시간**으로 가운데 50%가 이 범위에 분포했다. 사분위범위(IQR)는 **147.8시간**, 표준편차는 **117.3시간**으로 컨테이너별 체류시간 편차가 나타났다.

### 3. 장기 체류 컨테이너 발생 특성

최솟값은 **1.9시간**, 최댓값은 **885.0시간(약 36.9일)**으로 최대 체류시간은 평균의 약 **4.5배**였다. 장기 체류의 원인은 이 통계만으로 판단할 수 없다.


## 4. HJNC 자부두 환적 컨테이너의 야드 체류시간 구간별 분포 분석

야드 체류시간이 **어느 시간 구간에 집중되는지** 확인하여 체류 현황을 파악한다. 구간별 분포는 현황 설명용이며, 이후 상관분석에서는 실제 체류시간의 차이를 유지하기 위해 연속형 `야드체류시간(h)`을 사용한다.


In [20]:


import numpy as np
import matplotlib.pyplot as plt

col = "야드체류시간(h)"

yard_valid = df[
    df[col].notna() & (df[col] > 0)
].copy()

#  체류시간 구간 설정
bins = [0, 24, 48, 72, 120, np.inf]

labels = [
    "24시간 미만",
    "24~48시간",
    "48~72시간",
    "72~120시간",
    "120시간 이상"
]

yard_valid["체류시간구간"] = pd.cut(
    yard_valid[col],
    bins=bins,
    labels=labels,
    right=False,
    include_lowest=True
)

#  구간별 컨테이너 수 및 비율
yard_dist = (
    yard_valid["체류시간구간"]
    .value_counts(sort=False)
    .rename_axis("체류시간구간")
    .reset_index(name="컨테이너수")
)

yard_dist["비율(%)"] = (
    yard_dist["컨테이너수"] / len(yard_valid) * 100
).round(1)

print("========== 야드 체류시간 구간별 분포 ==========")
display(yard_dist)

#  장기 체류 현황
long_stay = pd.DataFrame({
    "구분": ["72시간 이상", "120시간 이상"],
    "컨테이너수": [
        (yard_valid[col] >= 72).sum(),
        (yard_valid[col] >= 120).sum()
    ]
})

long_stay["비율(%)"] = (
    long_stay["컨테이너수"] / len(yard_valid) * 100
).round(1)

print("\n========== 장기 체류 현황 ==========")
display(long_stay)

========== 야드 체류시간 구간별 분포 ==========


,체류시간구간,컨테이너수,비율(%)
0,24시간 미만,40,2.7
1,24~48시간,51,3.4
2,48~72시간,103,6.9
3,72~120시간,203,13.6
4,120시간 이상,1093,73.4



========== 장기 체류 현황 ==========


,구분,컨테이너수,비율(%)
0,72시간 이상,1296,87.0
1,120시간 이상,1093,73.4


### 1. 주요 체류시간 구간

| 체류시간 | 컨테이너 수 | 비율 |
|:---|---:|---:|
| 24시간 미만 | 40 | 2.7% |
| 24~48시간 | 51 | 3.4% |
| 48~72시간 | 103 | 6.9% |
| 72~120시간 | 203 | 13.6% |
| **120시간 이상** | **1,093** | **73.4%** |
| **합계** | **1,490** | **100.0%** |

**120시간 이상 구간이 1,093건(73.4%)으로 가장 높은 비중** 을 차지했다. 

전체 컨테이너의 약 4분의 3이 야드 장치 후 온샤시 처리까지 5일 이상 소요된 것으로 나타났다.

### 2. 단기 체류(48시간 미만)

**91건(6.1%)** 으로, 야드 장치 후 2일 이내에 온샤시 처리된 컨테이너의 비중은 낮았다. 

반면 **48시간 이상은 1,399건(93.9%)** 으로 대부분을 차지했다.

### 3. 장기 체류(72시간 이상)

**1,296건(87.0%)** 이며, 이 중 **203건(13.6%)** 은 72~120시간, **1,093건(73.4%)** 은 120시간 이상이었다. 

장기 체류는 일부 극단 사례만의 현상이 아니지만, 환적 선박 연결 일정에 따른 정상 대기가 포함될 수 있으므로 곧바로 운영 지연이나 병목으로 해석할 수 없다.

**분석 결과 및 다음 단계:** 높은 장기 체류 비중을 확인했지만, 구간별 분포만으로는 컨테이너별 체류시간 차이와 관련된 요인을 파악하기 어렵다. 

따라서 다음 단계에서는 **적하모선항차를 선석스케줄과 연결** 하여 선박 운영시간과 **개별 컨테이너의 야드 체류시간**  사이에 관계가 있는지 검토한다.


## 5. HJNC 자부두 환적 컨테이너와 선석스케줄 데이터 연결 분석

```
1. 선석스케줄 데이터 확인
```

In [21]:


schedule_candidates = [folder / name for folder in data_dirs for name in (
    "HJNC_선석스케줄(4).csv", "HJNC_선석스케줄.csv"
)]
schedule_path = next((p for p in schedule_candidates if p.exists()), None)
if schedule_path is None:
    raise FileNotFoundError(
        "HJNC_선석스케줄(4).csv 파일을 노트북과 같은 폴더 또는 "
        r"C:\project\data\hjnc_data 에 놓아주세요."
    )

schedule = pd.read_csv(
    schedule_path,
    encoding="utf-8-sig",
    low_memory=False
)

print("========== 선석스케줄 데이터 ==========")
print("전체 데이터:", len(schedule))
print("전체 컬럼 수:", len(schedule.columns))

print("\n컬럼 목록")
print(schedule.columns.tolist())

print("\n상위 5개 데이터")
display(schedule.head())

print("\n모선항차 중복 건수")
print(schedule["모선항차"].duplicated().sum())

========== 선석스케줄 데이터 ==========
전체 데이터: 1002
전체 컬럼 수: 18

컬럼 목록
['No', '선석', '항로', '모선항차', '선박명', '선사항차', '접안', '선사', '반입 시작일시', '반입 마감일시', '입항일시', '출항일시', '작업 시작일시', '작업 완료일시', '양하', '선적', 'S/H', '전배']

상위 5개 데이터


,No,선석,항로,모선항차,선박명,선사항차,접안,선사,반입 시작일시,반입 마감일시,입항일시,출항일시,작업 시작일시,작업 완료일시,양하,선적,S/H,전배
0,1,3B,FE4,HOHA-2024-0003,HMM HAMBURG,015W-015W,S,HMM,2024-12-27 00:00,2024-12-30 12:00,2024-12-30 22:30,2025-01-02 07:00,2024-12-30 23:30,2025-01-02 06:20,4766,2299,458,NaN
1,2,1B,MD3,YWDM-2024-0002,YM WISDOM,020W-021W,S,YML,2024-12-28 00:00,2024-12-31 06:00,2024-12-31 15:54,2025-01-02 23:00,2024-12-31 17:30,2025-01-02 21:46,3122,1276,2166,NaN
2,3,2B,JVH,OQPR-2024-0017,PEARL RIVER BRIDGE,207S-207S,S,ONE,2024-12-28 00:00,2024-12-31 07:00,2024-12-31 17:00,2025-01-01 05:00,2024-12-31 18:10,2025-01-01 04:31,308,398,20,NaN
3,4,2B,BH3,BIGE-2025-0001,BIG GEORGE,25001W-25001W,S,BEN,2024-12-30 00:00,2025-01-01 23:00,2025-01-02 09:15,2025-01-03 13:00,2025-01-02 09:40,2025-01-03 12:47,1105,1135,0,NaN
4,5,4B,PN3E,YTRG-2025-0001,YM TARGET,022E-022E,S,YML,2024-12-30 00:00,2025-01-01 23:00,2025-01-02 09:10,2025-01-03 18:00,2025-01-02 09:10,2025-01-03 17:50,1234,2368,12,NaN



모선항차 중복 건수
0


```
2. 선석스케줄 불러오기
```

In [22]:

schedule = pd.read_csv(
    schedule_path,
    encoding="utf-8-sig",
    low_memory=False
)

print("선석스케줄 전체:", len(schedule))
print("모선항차 중복:", schedule["모선항차"].duplicated().sum())

display(schedule.head())

display(
    schedule[
        [
            "모선항차",
            "선박명",
            "선석",
            "입항일시",
            "출항일시",
            "작업 시작일시",
            "작업 완료일시"
        ]
    ].head()
)

선석스케줄 전체: 1002
모선항차 중복: 0


,No,선석,항로,모선항차,선박명,선사항차,접안,선사,반입 시작일시,반입 마감일시,입항일시,출항일시,작업 시작일시,작업 완료일시,양하,선적,S/H,전배
0,1,3B,FE4,HOHA-2024-0003,HMM HAMBURG,015W-015W,S,HMM,2024-12-27 00:00,2024-12-30 12:00,2024-12-30 22:30,2025-01-02 07:00,2024-12-30 23:30,2025-01-02 06:20,4766,2299,458,NaN
1,2,1B,MD3,YWDM-2024-0002,YM WISDOM,020W-021W,S,YML,2024-12-28 00:00,2024-12-31 06:00,2024-12-31 15:54,2025-01-02 23:00,2024-12-31 17:30,2025-01-02 21:46,3122,1276,2166,NaN
2,3,2B,JVH,OQPR-2024-0017,PEARL RIVER BRIDGE,207S-207S,S,ONE,2024-12-28 00:00,2024-12-31 07:00,2024-12-31 17:00,2025-01-01 05:00,2024-12-31 18:10,2025-01-01 04:31,308,398,20,NaN
3,4,2B,BH3,BIGE-2025-0001,BIG GEORGE,25001W-25001W,S,BEN,2024-12-30 00:00,2025-01-01 23:00,2025-01-02 09:15,2025-01-03 13:00,2025-01-02 09:40,2025-01-03 12:47,1105,1135,0,NaN
4,5,4B,PN3E,YTRG-2025-0001,YM TARGET,022E-022E,S,YML,2024-12-30 00:00,2025-01-01 23:00,2025-01-02 09:10,2025-01-03 18:00,2025-01-02 09:10,2025-01-03 17:50,1234,2368,12,NaN


,모선항차,선박명,선석,입항일시,출항일시,작업 시작일시,작업 완료일시
0,HOHA-2024-0003,HMM HAMBURG,3B,2024-12-30 22:30,2025-01-02 07:00,2024-12-30 23:30,2025-01-02 06:20
1,YWDM-2024-0002,YM WISDOM,1B,2024-12-31 15:54,2025-01-02 23:00,2024-12-31 17:30,2025-01-02 21:46
2,OQPR-2024-0017,PEARL RIVER BRIDGE,2B,2024-12-31 17:00,2025-01-01 05:00,2024-12-31 18:10,2025-01-01 04:31
3,BIGE-2025-0001,BIG GEORGE,2B,2025-01-02 09:15,2025-01-03 13:00,2025-01-02 09:40,2025-01-03 12:47
4,YTRG-2025-0001,YM TARGET,4B,2025-01-02 09:10,2025-01-03 18:00,2025-01-02 09:10,2025-01-03 17:50


```
3. HJNC 컨테이너와 선석스케줄 모선항차 매칭
```

In [23]:


#  컨테이너 적하모선항차 변환
def convert_voyage(value):
    if pd.isna(value):
        return None

    value = str(value).strip()

    try:
        code, rest = value.split("-", 1)
        number, year = rest.split("/")

        return f"{code}-{year}-{int(number):04d}"
    except (ValueError, TypeError):
        return None


#  원본 데이터 유지하면서 매칭용 컬럼 생성
container_match = df.copy()

container_match["매칭용_모선항차"] = (
    container_match["적하모선항차"].apply(convert_voyage)
)

schedule_match = schedule.copy()

schedule_match["매칭용_모선항차"] = (
    schedule_match["모선항차"].astype("string").str.strip()
)


#  선석스케줄 모선항차와 비교
matched = container_match["매칭용_모선항차"].isin(
    schedule_match["매칭용_모선항차"]
)

print("========== 모선항차 매칭 결과 ==========")
print("전체 컨테이너:", len(container_match))
print("매칭 성공:", matched.sum())
print("매칭 실패:", (~matched).sum())
print("매칭률:", round(matched.mean() * 100, 1), "%")

print("\n========== 매칭 성공 상위 예시 ==========")
display(
    container_match.loc[
        matched,
        ["컨테이너번호", "적하모선항차",
         "매칭용_모선항차", "야드체류시간(h)"]
    ].head()
)




========== 모선항차 매칭 결과 ==========
전체 컨테이너: 1490
매칭 성공: 1490
매칭 실패: 0
매칭률: 100.0 %

========== 매칭 성공 상위 예시 ==========


,컨테이너번호,적하모선항차,매칭용_모선항차,야드체류시간(h)
0,AMFU8667185,SMNB-007/2025,SMNB-2025-0007,476.9
1,AXIU1350376,SMYT-005/2025,SMYT-2025-0005,74.6
2,AXIU4223734,SMSH-012/2025,SMSH-2025-0012,107.4
3,BEAU4507198,HODU-002/2025,HODU-2025-0002,129.5
4,BEAU4549233,YIPT-001/2025,YIPT-2025-0001,101.9


컨테이너와 선석스케줄의 모선항차 표기(`SMNB-007/2025` ↔ `SMNB-2025-0007`)를 표준화했다. 

**1,490건 모두 일치해 매칭률 100.0%* , 미매칭은 0건이었다. 이는 적하모선항차가 선석스케줄에 존재한다는 의미이다.


```
4. 선석스케줄 정보 병합
```

In [24]:

schedule_cols = [
    "매칭용_모선항차",
    "선박명",
    "선석",
    "입항일시",
    "출항일시",
    "작업 시작일시",
    "작업 완료일시",
    "반입 시작일시",
    "반입 마감일시"
]

#  컨테이너 기준으로 병합
merged_df = container_match.merge(
    schedule_match[schedule_cols],
    on="매칭용_모선항차",
    how="left",
    validate="many_to_one",
    suffixes=("", "_선석")
)


print("========== 선석스케줄 병합 결과 ==========")
print("병합 전 컨테이너:", len(container_match))
print("병합 후 컨테이너:", len(merged_df))

print(
    "선박명 연결 성공:",
    merged_df["선박명_선석"].notna().sum()
    if "선박명_선석" in merged_df.columns
    else merged_df["선박명"].notna().sum()
)


display(
    merged_df[
        [
            "컨테이너번호",
            "적하모선항차",
            "매칭용_모선항차",
            "야드체류시간(h)",
            "선석",
            "입항일시",
            "작업 시작일시",
            "작업 완료일시",
            "출항일시"
        ]
    ].head()
)

========== 선석스케줄 병합 결과 ==========
병합 전 컨테이너: 1490
병합 후 컨테이너: 1490
선박명 연결 성공: 1490


,컨테이너번호,적하모선항차,매칭용_모선항차,야드체류시간(h),선석,입항일시,작업 시작일시,작업 완료일시,출항일시
0,AMFU8667185,SMNB-007/2025,SMNB-2025-0007,476.9,3B,2025-05-30 23:32,2025-05-30 23:50,2025-05-31 17:12,2025-05-31 18:00
1,AXIU1350376,SMYT-005/2025,SMYT-2025-0005,74.6,1B,2025-05-04 14:05,2025-05-04 15:05,2025-05-05 05:13,2025-05-05 06:00
2,AXIU4223734,SMSH-012/2025,SMSH-2025-0012,107.4,1B,2025-09-12 09:45,2025-09-12 10:00,2025-09-12 22:05,2025-09-12 23:00
3,BEAU4507198,HODU-002/2025,HODU-2025-0002,129.5,3B,2025-09-30 15:39,2025-09-30 15:40,2025-10-03 07:05,2025-10-03 08:00
4,BEAU4549233,YIPT-001/2025,YIPT-2025-0001,101.9,1B,2025-02-15 01:03,2025-02-15 02:20,2025-02-16 08:15,2025-02-16 10:00


모선항차 기준 병합 결과, **컨테이너 1,490건 모두 선박 정보가 연결** 되었고 병합 전후 행 수가 유지됐다.

 동일 적하 항차의 컨테이너는 같은 선박 시간 정보를 공유하므로, 이후 상관분석에서는 반복 관측에 유의한다.


## 6. 선박 시간 지표와 야드 체류시간의 관계 분석


**분석 목적:** 4번에서 확인한 장기 체류 현상에 대해, 선박 운영시간의 길이가 개별 컨테이너 야드 체류시간과 연관되는지 탐색한다. 

현재 선박의 시간 지표를 먼저 검토하고, 이후 **동일 선석에서 직전에 입항한 선박**까지 분석 범위를 확장한다. 

관련성이 약하더라도 이를 운영 영향의 부재나 인과관계의 부정으로 단정하지 않는다.

**분석 순서:** 6-1 선박 시간 지표 계산 → 6-2 현재 선박과 야드 체류시간 상관분석 → 6-3 직전 입항 선박과 후속 선박·컨테이너 관계 → 6-3 종합 결론 및 향후 시뮬레이션 방향.


### 6-1. 선박 시간 지표 5개 계산

In [25]:




analysis_df = merged_df.copy()


time_cols = [
    "반입 시작일시",
    "반입 마감일시",
    "입항일시",
    "출항일시",
    "작업 시작일시",
    "작업 완료일시"
]

for col in time_cols:
    analysis_df[col] = pd.to_datetime(
        analysis_df[col],
        errors="coerce"
    )

#  선박 시간 지표 계산 (시간 단위)
time_metrics = {
    "입항 지연시간": ("입항일시", "반입 시작일시"),
    "작업시작전 대기시간": ("작업 시작일시", "입항일시"),
    "작업시간": ("작업 완료일시", "작업 시작일시"),
    "작업 후 출항 대기시간": ("출항일시", "작업 완료일시"),
    "출항 지연시간": ("출항일시", "반입 마감일시")
}

for metric, (end, start) in time_metrics.items():
    analysis_df[metric] = (
        (analysis_df[end] - analysis_df[start])
        .dt.total_seconds() / 3600
    ).round(1)


print("========== 선박 시간 지표 계산 결과 ==========")
print("전체 컨테이너:", len(analysis_df))

validation = pd.DataFrame({
    "결측치": analysis_df[list(time_metrics)].isna().sum(),
    "음수 건수": (analysis_df[list(time_metrics)] < 0).sum()
})

display(validation)


summary = (
    analysis_df[list(time_metrics)]
    .describe()
    .round(1)
)

display(summary)


display(
    analysis_df[
        ["컨테이너번호", "적하모선항차", "야드체류시간(h)"]
        + list(time_metrics)
    ].head()
)

========== 선박 시간 지표 계산 결과 ==========
전체 컨테이너: 1490


,결측치,음수 건수
입항 지연시간,0,0
작업시작전 대기시간,0,0
작업시간,0,0
작업 후 출항 대기시간,0,0
출항 지연시간,0,0


,입항 지연시간,작업시작전 대기시간,작업시간,작업 후 출항 대기시간,출항 지연시간
count,1490.0,1490.0,1490.0,1490.0,1490.0
mean,83.4,0.7,21.1,1.1,33.1
std,6.9,1.5,11.2,0.7,11.4
min,71.9,0.0,0.9,0.0,16.0
25%,78.3,0.0,13.7,0.7,26.0
50%,83.4,0.7,16.4,1.0,28.0
75%,88.8,1.0,25.7,1.4,37.2
max,95.7,26.9,67.2,5.1,83.0


,컨테이너번호,적하모선항차,야드체류시간(h),입항 지연시간,작업시작전 대기시간,작업시간,작업 후 출항 대기시간,출항 지연시간
0,AMFU8667185,SMNB-007/2025,476.9,95.5,0.3,17.4,0.8,29.0
1,AXIU1350376,SMYT-005/2025,74.6,86.1,1.0,14.1,0.8,26.0
2,AXIU4223734,SMSH-012/2025,107.4,81.8,0.2,12.1,0.9,23.0
3,BEAU4507198,HODU-002/2025,129.5,87.6,0.0,63.4,0.9,75.0
4,BEAU4549233,YIPT-001/2025,101.9,73.0,1.3,29.9,1.8,43.0


#### 6-1. 시간 지표 계산 결과 및 해석

5개 선박 시간 지표의 **결측치·음수는 모두 0건**이었다. 

평균은 입항 지연시간(대체지표) **83.4h**, 작업시작전 대기시간 **0.7h**, 작업시간 **21.1h**, 작업 후 출항 대기시간 **1.1h**, 출항 지연시간(대체지표) **33.1h**였다.

이는 **컨테이너 1,490건 기준**으로 집계한 결과이며, 동일 항차의 컨테이너는 같은 선박 시간 값을 공유한다.

 입·출항 지연시간은 각각 `입항일시 − 반입 시작일시`, `출항일시 − 반입 마감일시`로 계산한 **반입 일정 기준 경과시간(대체지표)** 이며 실제 예정시각 대비 지연을 뜻하지 않는다.


### 6-2. 기존 선박 시간 지표와 야드 체류시간의 상관분석

| 선박 시간 지표 | 확인할 내용 |
|:---|:---|
| 입항 지연시간 | 반입 시작부터 선박 입항까지의 경과시간과 야드 체류시간의 관계 |
| 작업시작전 대기시간 | 입항 후 작업 시작까지의 경과시간과 야드 체류시간의 관계 |
| 작업시간 | 선박 작업시간과 야드 체류시간의 관계 |
| 작업 후 출항 대기시간 | 작업 완료 후 출항까지의 경과시간과 야드 체류시간의 관계 |
| 출항 지연시간 | 반입 마감부터 출항까지의 경과시간과 야드 체류시간의 관계 |

**용어 주의:** 이 노트북의 `입항 지연시간`은 `입항일시 − 반입 시작일시`, `출항 지연시간`은 `출항일시 − 반입 마감일시`로 계산한다. 

두 값 모두 예정 입·출항시각 대비 실제 지연시간이 아니라 **반입 일정 기준 경과시간(대체 지표)** 이다.

In [26]:




target = "야드체류시간(h)"

metrics = [
    "입항 지연시간",
    "작업시작전 대기시간",
    "작업시간",
    "작업 후 출항 대기시간",
    "출항 지연시간"
]

#  지표별 피어슨 상관계수 계산
results = []

for metric in metrics:
    valid = analysis_df[[target, metric]].dropna()

    corr = valid[target].corr(valid[metric])

    results.append({
        "선박 시간 지표": metric,
        "분석 컨테이너수": len(valid),
        "상관계수(r)": round(corr, 4)
    })

corr_df = pd.DataFrame(results)

print("========== 야드 체류시간과 선박 시간 지표 상관분석 ==========")
display(corr_df)

# 상관계수 절댓값 기준 정렬
corr_sorted = corr_df.copy()

corr_sorted["절댓값"] = corr_sorted["상관계수(r)"].abs()

corr_sorted = (
    corr_sorted
    .sort_values("절댓값", ascending=False)
    .drop(columns="절댓값")
    .reset_index(drop=True)
)

print("\n========== 상관관계 크기순 정렬 ==========")
display(corr_sorted)

========== 야드 체류시간과 선박 시간 지표 상관분석 ==========


,선박 시간 지표,분석 컨테이너수,상관계수(r)
0,입항 지연시간,1490,0.0620
1,작업시작전 대기시간,1490,0.1124
2,작업시간,1490,0.0920
3,작업 후 출항 대기시간,1490,-0.0317
4,출항 지연시간,1490,0.1038



========== 상관관계 크기순 정렬 ==========


,선박 시간 지표,분석 컨테이너수,상관계수(r)
0,작업시작전 대기시간,1490,0.1124
1,출항 지연시간,1490,0.1038
2,작업시간,1490,0.0920
3,입항 지연시간,1490,0.0620
4,작업 후 출항 대기시간,1490,-0.0317


#### 6-2. 양의 상관관계를 보인 지표의 산점도 및 회귀선

In [27]:


import plotly.graph_objects as go
from plotly.subplots import make_subplots


target = "야드체류시간(h)"

metrics = [
    "작업시간",
    "출항 지연시간",
    "입항 지연시간",
    "작업시작전 대기시간"
]


fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=metrics,
    horizontal_spacing=0.13,
    vertical_spacing=0.23
)

#  산점도 및 회귀선
for i, metric in enumerate(metrics):

    row = i // 2 + 1
    col = i % 2 + 1

    valid = analysis_df[
        ["컨테이너번호", metric, target]
    ].dropna()

    x = valid[metric].to_numpy(dtype=float)
    y = valid[target].to_numpy(dtype=float)

    # 피어슨 상관계수
    r = valid[metric].corr(valid[target])

    # 선형 회귀선
    slope, intercept = np.polyfit(x, y, 1)

    x_line = np.linspace(x.min(), x.max(), 100)
    y_line = slope * x_line + intercept

    # 산점도
    fig.add_trace(
        go.Scatter(
            x=x,
            y=y,
            mode="markers",
            marker=dict(
                size=6,
                opacity=0.35,
                color="#4682B4"
            ),
            customdata=valid[["컨테이너번호"]].to_numpy(),
            hovertemplate=(
                "컨테이너번호: %{customdata[0]}<br>"
                f"{metric}: %{{x:.1f}}시간<br>"
                "야드 체류시간: %{y:.1f}시간"
                "<extra></extra>"
            ),
            showlegend=False
        ),
        row=row,
        col=col
    )

    # 회귀선
    fig.add_trace(
        go.Scatter(
            x=x_line,
            y=y_line,
            mode="lines",
            line=dict(
                color="red",
                width=3
            ),
            hovertemplate=(
                f"{metric} 회귀선<br>"
                "X: %{x:.1f}시간<br>"
                "추정 야드 체류시간: %{y:.1f}시간"
                "<extra></extra>"
            ),
            showlegend=False
        ),
        row=row,
        col=col
    )

   
    axis_num = "" if i == 0 else str(i + 1)

    fig.add_annotation(
        x=0.97,
        y=0.97,
        xref=f"x{axis_num} domain",
        yref=f"y{axis_num} domain",
        text=f"<b>r = {r:.4f}</b>",
        showarrow=False,
        xanchor="right",
        yanchor="top",
        font=dict(
            size=14,
            color="black"
        ),
        bgcolor="rgba(255,255,255,0.9)",
        bordercolor="lightgray",
        borderwidth=1,
        borderpad=5
    )

    #  X축 설명
    fig.update_xaxes(
        title_text=f"{metric} (h)",
        title_font=dict(size=13),
        tickfont=dict(size=11),
        showgrid=True,
        gridcolor="#E8EEF5",
        row=row,
        col=col
    )

    #  Y축 설명
    fig.update_yaxes(
        title_text="컨테이너 야드 체류시간(h)",
        title_font=dict(size=13),
        tickfont=dict(size=11),
        showgrid=True,
        gridcolor="#E8EEF5",
        row=row,
        col=col
    )


fig.update_layout(
    title=dict(
        text="HJNC 자부두 환적: 선박 시간 지표와 야드 체류시간의 상관관계",
        x=0.5,
        font=dict(size=20)
    ),
    width=1250,
    height=950,
    template="plotly_white",
    font=dict(
        family="Malgun Gothic",
        size=12
    ),
    margin=dict(
        t=100,
        b=80,
        l=90,
        r=50
    )
)

fig.show()

#### 6-2. 산점도의 구성과 해석

X축은 선박 시간 지표, Y축은 **컨테이너 야드 체류시간**이다. 파란 점은 실제 기록, 빨간 선은 선형 회귀선, `r`은 피어슨 상관계수이다.

 양의 상관계수를 보인 네 지표 모두 **r=0.0620~0.1124**로, 회귀선이 완만하게 상승하더라도 뚜렷한 선형 관계는 확인하기 어렵다.


**1. 작업시작전 대기시간과 야드 체류시간**

피어슨 **r=0.1124**로 양의 상관계수를 보인 4개 지표 중 가장 높았다. 

5시간 초과 관측값 **7건**을 제외한 민감도 분석에서는 **r=0.0861**로 감소했으나, 제외 전후 모두 선형 관계는 매우 약했다.

 5시간은 이상치 확정 기준이 아닌 비교를 위한 임시 기준이다.

**2. 출항 지연시간(대체지표)과 야드 체류시간**

피어슨 **r=0.1038**로 두 번째로 높았다. `출항일시 − 반입 마감일시`로 계산한 **경과시간**이므로 실제 출항 지연과 구분한다.

**3. 나머지 지표**

작업시간 **r=0.0920**, 입항 지연시간(대체지표) **r=0.0620**으로 모두 매우 약한 양의 상관관계를 보였다. 

산점도에서도 관측값이 넓게 분산됐다. 산점도에 포함하지 않은 **작업 후 출항 대기시간은 r=-0.0317**이었다.


**종합 분석:**
- 5개 지표 모두 상관계수의 절댓값이 **0.12 미만**으로, 선박 시간 지표와 야드 체류시간 사이에 뚜렷한 선형 관계가 확인되지 않았다. 

- 동일 항차의 선박 시간이 반복 반영된 **컨테이너 기준 탐색적 결과**로 해석한다.


#### 6-2. 상관분석 정리

- 작업시작전 대기시간(**r=0.1124**)과 출항 지연시간 대체지표(**r=0.1038**)가 상대적으로 높았으나 둘 다 매우 약한 양의 상관관계였다.
  
- 작업시간(**r=0.0920**), 입항 지연시간 대체지표(**r=0.0620**), 작업 후 출항 대기시간(**r=-0.0317**)도 뚜렷한 선형 관계를 보이지 않았다.
  
- 입·출항 지연시간은 실제 예정시각 대비 지연이 아니라 **반입 일정 기준 경과시간**이다.

**다음:** 현재 선박 지표만으로 뚜렷한 관계를 확인하기 어려워, 동일 선석의 직전 입항 선박 운영시간과 후속 선박·컨테이너의 관계로 탐색 범위를 확장한다.


### 6-3. 동일 선석의 이전 선박과 후속 선박의 관계 분석

6-2에서는 현재 적하 선박의 운영시간과 컨테이너 야드 체류시간 사이에 뚜렷한 선형 관계를 확인하기 어려웠다. 

이에 따라 **동일 선석의 직전 입항 항차**를 연결하여, 직전 선박의 운영시간과 선박 간 시간 간격이 후속 선박의 작업시작전 대기시간 및 후속 컨테이너 야드 체류시간과 관련되는지 검토한다.

`출항일시 − 입항일시`는 실제 접안·이안시각에 근거한 선석 점유시간이 아니라 **입항~출항 경과시간을 이용한 대체지표**이다. 직전 항차는 실제 접안 순서가 아니라 **동일 선석 내 입항 기록 순서**로 결정한다.





| 분석 지표 | 계산식 | 확인할 내용 |
|:---|:---|:---|
| **이전 선박 작업시간** | 이전 선박 작업 완료 − 작업 시작 | 직전 선박의 작업 소요시간 |
| **이전 선박 선석 점유시간(대체 지표)** | 이전 선박 출항 − 입항 | 직전 선박의 입항~출항 경과시간 |
| **선박 간 시간 간격** | 현재 선박 입항 − 이전 선박 출항 | 동일 선석에서 두 선박의 시간 간격 |
| **현재 선박 작업시작전 대기시간** | 현재 선박 작업 시작 − 입항 | 현재 선박의 입항 이후 작업 시작까지의 시간 |
| **현재 컨테이너 야드 체류시간** | 온샤시 − 야드 장치 | 후속 선박에 적하되는 컨테이너의 체류시간 |


#### 6-3-1. 선석스케줄 데이터 및 시간 정합성 검증

In [28]:



berth_df = schedule.copy()


time_cols = [
    "입항일시",
    "출항일시",
    "작업 시작일시",
    "작업 완료일시"
]

for col in time_cols:
    berth_df[col] = pd.to_datetime(
        berth_df[col],
        errors="coerce"
    )


print("========== 선석스케줄 기본 정보 ==========")
print("전체 선박 항차:", len(berth_df))
print("선석 종류:", berth_df["선석"].nunique())

print("\n========== 선석별 선박 항차 수 ==========")
display(
    berth_df["선석"]
    .value_counts(dropna=False)
    .rename_axis("선석")
    .reset_index(name="선박 항차 수")
)


print("\n========== 결측치 검증 ==========")
display(
    berth_df[
        ["선석", "모선항차"] + time_cols
    ].isna().sum().to_frame("결측치")
)


berth_df = berth_df.sort_values(
    ["선석", "입항일시", "작업 시작일시"]
).reset_index(drop=True)

print("\n========== 선석별 선박 배치 확인 ==========")
display(
    berth_df[
        [
            "선석",
            "모선항차",
            "선박명",
            "입항일시",
            "작업 시작일시",
            "작업 완료일시",
            "출항일시"
        ]
    ].head(20)
)


print("\n========== 시간 순서 검증 ==========")

checks = {
    "입항 이후 출항 이전": ("출항일시", "입항일시"),
    "작업 시작 이후 완료": ("작업 완료일시", "작업 시작일시"),
    "입항 이후 작업 시작": ("작업 시작일시", "입항일시"),
    "작업 완료 이후 출항": ("출항일시", "작업 완료일시")
}

for name, (end, start) in checks.items():
    valid_rows = berth_df[end].notna() & berth_df[start].notna()
    violations = valid_rows & (berth_df[end] < berth_df[start])
    print(f"{name} 위반: {violations.sum()}건 (비교 불가: {(~valid_rows).sum()}건)")


========== 선석스케줄 기본 정보 ==========


전체 선박 항차: 1002
선석 종류: 4

========== 선석별 선박 항차 수 ==========


,선석,선박 항차 수
0,2B,385
1,1B,296
2,3B,179
3,4B,142



========== 결측치 검증 ==========


,결측치
선석,0
모선항차,0
입항일시,0
출항일시,0
작업 시작일시,1
작업 완료일시,0



========== 선석별 선박 배치 확인 ==========


,선석,모선항차,선박명,입항일시,작업 시작일시,작업 완료일시,출항일시
0,1B,YWDM-2024-0002,YM WISDOM,2024-12-31 15:54:00,2024-12-31 17:30:00,2025-01-02 21:46:00,2025-01-02 23:00:00
1,1B,ZGHZ-2025-0001,ZHONG GU HANG ZHOU,2025-01-03 00:55:00,2025-01-03 14:10:00,2025-01-03 21:42:00,2025-01-03 22:00:00
2,1B,OSWR-2025-0001,SWAN RIVER BRIDGE,2025-01-04 00:05:00,2025-01-04 00:53:00,2025-01-04 10:59:00,2025-01-04 11:00:00
3,1B,KKJA-2025-0001,KMTC JEBEL ALI,2025-01-04 13:50:00,2025-01-04 14:50:00,2025-01-05 02:23:00,2025-01-05 03:00:00
4,1B,HOPD-2025-0001,HMM PERIDOT,2025-01-05 06:05:00,2025-01-05 07:04:00,2025-01-07 03:40:00,2025-01-07 05:01:00
5,1B,OZEU-2025-0001,ZEUS LUMOS,2025-01-07 07:26:00,2025-01-07 08:00:00,2025-01-08 14:47:00,2025-01-08 15:00:00
6,1B,YWND-2025-0001,YM WONDROUS,2025-01-08 17:40:00,2025-01-08 17:40:00,2025-01-10 00:38:00,2025-01-10 01:00:00
7,1B,OONM-2025-0001,ONE MARVEL,2025-01-10 03:10:00,2025-01-10 03:11:00,2025-01-11 08:59:00,2025-01-11 09:00:00
8,1B,OZEL-2025-0001,ZENITH LUMOS,2025-01-11 11:45:00,2025-01-11 12:45:00,2025-01-12 12:59:00,2025-01-12 14:00:00
9,1B,HODM-2025-0001,HMM DIAMOND,2025-01-12 16:35:00,2025-01-12 16:35:00,2025-01-13 15:39:00,2025-01-13 17:00:00



========== 시간 순서 검증 ==========
입항 이후 출항 이전 위반: 0건 (비교 불가: 0건)
작업 시작 이후 완료 위반: 0건 (비교 불가: 1건)
입항 이후 작업 시작 위반: 0건 (비교 불가: 1건)
작업 완료 이후 출항 위반: 0건 (비교 불가: 0건)


전체 **1,002개 항차**는 선석 4개(1B 296건, 2B 385건, 3B 179건, 4B 142건)에 분포했다. **작업 시작일시 결측 1건**이 확인됐다. 결측으로 인한 비교 불가는 실제 시간 순서 위반과 구분해야 한다.


#### 6-3-2. 동일 선석의 이전 선박과 현재 선박 연결

In [29]:





berth_analysis_df = berth_df.copy()


print("========== 작업 시작일시 결측 항차 ==========")

display(
    berth_analysis_df[
        berth_analysis_df["작업 시작일시"].isna()
    ][
        [
            "선석",
            "모선항차",
            "선박명",
            "입항일시",
            "작업 시작일시",
            "작업 완료일시",
            "출항일시"
        ]
    ]
)


berth_analysis_df = berth_analysis_df.sort_values(
    ["선석", "입항일시", "모선항차"]
).reset_index(drop=True)


previous_cols = [
    "모선항차",
    "선박명",
    "입항일시",
    "출항일시",
    "작업 시작일시",
    "작업 완료일시"
]

for col in previous_cols:
    berth_analysis_df[f"이전_{col}"] = (
        berth_analysis_df.groupby("선석")[col].shift(1)
    )

#  이전 선박 작업시간
berth_analysis_df["이전 선박 작업시간(h)"] = (
    (
        berth_analysis_df["이전_작업 완료일시"]
        - berth_analysis_df["이전_작업 시작일시"]
    ).dt.total_seconds() / 3600
).round(1)

#  이전 선박 입항~출항 경과시간
berth_analysis_df["이전 선박 점유시간(h)"] = (
    (
        berth_analysis_df["이전_출항일시"]
        - berth_analysis_df["이전_입항일시"]
    ).dt.total_seconds() / 3600
).round(1)

#  선박 간 시간 간격
berth_analysis_df["선박 간 시간 간격(h)"] = (
    (
        berth_analysis_df["입항일시"]
        - berth_analysis_df["이전_출항일시"]
    ).dt.total_seconds() / 3600
).round(1)

#  현재 선박 작업시작전 대기시간
berth_analysis_df["현재 선박 작업시작전 대기시간(h)"] = (
    (
        berth_analysis_df["작업 시작일시"]
        - berth_analysis_df["입항일시"]
    ).dt.total_seconds() / 3600
).round(1)

#  연결 결과 확인
print("\n========== 이전 선박 연결 결과 ==========")

print("전체 선박 항차:", len(berth_analysis_df))

print(
    "이전 선박 연결 건수:",
    berth_analysis_df["이전_모선항차"].notna().sum()
)

print(
    "이전 선박 없는 항차:",
    berth_analysis_df["이전_모선항차"].isna().sum()
)

#  선박 간 시간 간격 검증
gap = berth_analysis_df["선박 간 시간 간격(h)"]

print("\n========== 선박 간 시간 간격 검증 ==========")
print("양수:", (gap > 0).sum())
print("0:", (gap == 0).sum())
print("음수:", (gap < 0).sum())
print("결측치:", gap.isna().sum())

#  기초 통계
metrics = [
    "이전 선박 작업시간(h)",
    "이전 선박 점유시간(h)",
    "선박 간 시간 간격(h)",
    "현재 선박 작업시작전 대기시간(h)"
]

print("\n========== 선박 시간 지표 기초 통계 ==========")

display(
    berth_analysis_df[metrics]
    .describe()
    .round(1)
)


print("\n========== 동일 선석 이전 선박 연결 예시 ==========")

display(
    berth_analysis_df[
        [
            "선석",
            "이전_모선항차",
            "모선항차",
            "이전 선박 작업시간(h)",
            "이전 선박 점유시간(h)",
            "선박 간 시간 간격(h)",
            "현재 선박 작업시작전 대기시간(h)"
        ]
    ].dropna(subset=["이전_모선항차"]).head(15)
)

========== 작업 시작일시 결측 항차 ==========


,선석,모선항차,선박명,입항일시,작업 시작일시,작업 완료일시,출항일시
949,4B,BJBG-2025-0004,BEIJING BRIDGE,2025-08-22 16:00:00,NaT,2025-08-23 07:24:00,2025-08-23 11:00:00



========== 이전 선박 연결 결과 ==========
전체 선박 항차: 1002
이전 선박 연결 건수: 998
이전 선박 없는 항차: 4

========== 선박 간 시간 간격 검증 ==========
양수: 971
0: 0
음수: 27
결측치: 4

========== 선박 시간 지표 기초 통계 ==========


,이전 선박 작업시간(h),이전 선박 점유시간(h),선박 간 시간 간격(h),현재 선박 작업시작전 대기시간(h)
count,997.0,998.0,998.0,1001.0
mean,19.0,20.9,14.1,0.8
std,12.1,12.2,28.4,1.8
min,0.6,2.0,-21.0,0.0
25%,10.9,12.8,2.2,0.0
50%,15.6,17.6,2.9,0.5
75%,23.6,25.6,13.2,1.0
max,67.2,72.6,252.0,26.9



========== 동일 선석 이전 선박 연결 예시 ==========


,선석,이전_모선항차,모선항차,이전 선박 작업시간(h),이전 선박 점유시간(h),선박 간 시간 간격(h),현재 선박 작업시작전 대기시간(h)
1,1B,YWDM-2024-0002,ZGHZ-2025-0001,52.3,55.1,1.9,13.2
2,1B,ZGHZ-2025-0001,OSWR-2025-0001,7.5,21.1,2.1,0.8
3,1B,OSWR-2025-0001,KKJA-2025-0001,10.1,10.9,2.8,1.0
4,1B,KKJA-2025-0001,HOPD-2025-0001,11.6,13.2,3.1,1.0
5,1B,HOPD-2025-0001,OZEU-2025-0001,44.6,46.9,2.4,0.6
6,1B,OZEU-2025-0001,YWND-2025-0001,30.8,31.6,2.7,0.0
7,1B,YWND-2025-0001,OONM-2025-0001,31.0,31.3,2.2,0.0
8,1B,OONM-2025-0001,OZEL-2025-0001,29.8,29.8,2.8,1.0
9,1B,OZEL-2025-0001,HODM-2025-0001,24.2,26.2,2.6,0.0
10,1B,HODM-2025-0001,HOSO-2025-0001,23.1,24.4,2.2,0.0


##### 동일 선석 연결 및 시간 간격 결과

선석별 입항 순서로 정렬해 첫 항차 4건을 제외한 **998개 항차**에 직전 항차 정보를 연결했다.

 선박 간 시간 간격은 양수 **971건**, 음수 **27건**, 결측 **4건**이며 평균 **14.1h**, 중앙값 **2.9h**였다. 
 
 직전 선박 작업시간 평균은 **19.0h**, 입항~출항 경과시간 평균은 **20.9h**, 현재 선박의 작업시작전 대기시간 평균은 **0.8h**였다.

음수 간격 27건은 입출항 시각 기록이 중첩된 경우로, 실제 선석 중복 점유를 확정하지 않는다. 다음 단계에서는 직전 항차의 운영시간과 후속 선박 대기시간의 상관관계를 확인한다.


#### 6-3-3. 이전 선박 운영시간과 현재 선박 작업시작전 대기시간

직전 선박의 운영시간이 길수록 현재 선박의 입항 후 작업 시작까지의 경과시간도 길어지는지 **선박 항차 단위**로 검토한다.




 1. 분석 지표

| 독립변수(X축) | 종속변수(Y축) | 확인할 내용 |
|:---|:---|:---|
| **이전 선박 작업시간(h)** | 현재 선박 작업시작전 대기시간(h) | 이전 선박의 작업시간과 후속 선박 대기의 관계 |
| **이전 선박 점유시간(h)** | 현재 선박 작업시작전 대기시간(h) | 이전 선박의 입항~출항 경과시간과 후속 선박 대기의 관계 |
| **선박 간 시간 간격(h)** | 현재 선박 작업시작전 대기시간(h) | 선박 간 간격과 후속 선박 대기의 관계 |


In [30]:



berth_corr_data = berth_analysis_df.copy()

target = "현재 선박 작업시작전 대기시간(h)"

metrics = [
    "이전 선박 작업시간(h)",
    "이전 선박 점유시간(h)",
    "선박 간 시간 간격(h)"
]


print("========== 이전 선박 연결 현황 ==========")

print("전체 선박 항차:", len(berth_corr_data))

print(
    "이전 선박 연결 건수:",
    berth_corr_data["이전_모선항차"].notna().sum()
)

gap = berth_corr_data["선박 간 시간 간격(h)"]

print("선박 간 간격 음수:", (gap < 0).sum())
print("선박 간 간격 0:", (gap == 0).sum())
print("선박 간 간격 양수:", (gap > 0).sum())

#  피어슨 상관계수 계산
results = []

for metric in metrics:

    valid = berth_corr_data[
        [metric, target]
    ].replace([np.inf, -np.inf], np.nan).dropna()

    r = valid[metric].corr(
        valid[target],
        method="pearson"
    )

    results.append({
        "이전 선박 관련 지표": metric,
        "분석 선박 항차 수": len(valid),
        "상관계수(r)": round(r, 4)
    })

berth_corr_df = pd.DataFrame(results)


print("\n========== 선박 기준 피어슨 상관분석 ==========")

display(
    berth_corr_df.sort_values(
        "상관계수(r)",
        ascending=False
    )
)


print("\n========== 분석 지표 기초 통계 ==========")

display(
    berth_corr_data[
        metrics + [target]
    ].describe().round(1)
)


print("\n========== 선석별 분석 대상 ==========")

display(
    berth_corr_data[
        berth_corr_data["이전_모선항차"].notna()
    ]["선석"]
    .value_counts()
    .rename_axis("선석")
    .reset_index(name="연결 항차 수")
)

========== 이전 선박 연결 현황 ==========
전체 선박 항차: 1002
이전 선박 연결 건수: 998
선박 간 간격 음수: 27
선박 간 간격 0: 0
선박 간 간격 양수: 971

========== 선박 기준 피어슨 상관분석 ==========


,이전 선박 관련 지표,분석 선박 항차 수,상관계수(r)
1,이전 선박 점유시간(h),997,-0.0177
0,이전 선박 작업시간(h),996,-0.0196
2,선박 간 시간 간격(h),997,-0.0380



========== 분석 지표 기초 통계 ==========


,이전 선박 작업시간(h),이전 선박 점유시간(h),선박 간 시간 간격(h),현재 선박 작업시작전 대기시간(h)
count,997.0,998.0,998.0,1001.0
mean,19.0,20.9,14.1,0.8
std,12.1,12.2,28.4,1.8
min,0.6,2.0,-21.0,0.0
25%,10.9,12.8,2.2,0.0
50%,15.6,17.6,2.9,0.5
75%,23.6,25.6,13.2,1.0
max,67.2,72.6,252.0,26.9



========== 선석별 분석 대상 ==========


,선석,연결 항차 수
0,2B,384
1,1B,295
2,3B,178
3,4B,141


직전 선박 운영시간과 현재 선박 작업시작전 대기시간의 피어슨 상관계수는 **이전 작업시간 r=-0.0196**, **이전 입항~출항 경과시간(대체지표) r=-0.0177**, **선박 간 시간 간격 r=-0.0380**이었다. 

모두 0에 가까워 **뚜렷한 선형 관계가 확인되지 않았다.** 선박 간 간격 음수 **27건**의 해석에는 주의가 필요하다.

**다음:** 이번 결과의 종속변수는 후속 선박의 작업시작전 대기시간이었다. 이어서 프로젝트 핵심 지표인 **후속 컨테이너의 야드 체류시간**을 직접 비교한다.


#### 6-3-4. 이전 선박 운영시간과 후속 컨테이너 야드 체류시간

현재 선박과 직전 선박의 시간 지표를 분석한 결과를 이어서, **직전 선박의 운영시간과 후속 선박에 적하되는 컨테이너의 야드 체류시간**을 직접 비교한다. 

동일 항차 컨테이너의 반복 반영을 검토하기 위해 **컨테이너 기준과 항차 기준**을 함께 사용한다.

| 독립변수(X축) | 종속변수(Y축) | 확인할 내용 |
|:---|:---|:---|
| **이전 선박 작업시간(h)** | 후속 컨테이너 야드 체류시간(h) | 직전 선박의 작업시간과 체류시간의 관계 |
| **이전 선박 점유시간(h) (대체지표)** | 후속 컨테이너 야드 체류시간(h) | 직전 선박의 입항~출항 경과시간과 체류시간의 관계 |
| **선박 간 시간 간격(h)** | 후속 컨테이너 야드 체류시간(h) | 선박 간 간격과 체류시간의 관계 |


- 컨테이너 기준: 컨테이너 1건을 하나의 관측값으로 사용

- 선박 기준: 동일 항차에 속한 컨테이너들의 평균 야드 체류시간을 계산한 후 선박 1항차를 하나의 관측값으로 사용

In [31]:




container_df = analysis_df.copy()
vessel_df = berth_analysis_df.copy()



vessel_cols = [
    "모선항차",
    "선석",
    "이전_모선항차",
    "이전 선박 작업시간(h)",
    "이전 선박 점유시간(h)",
    "선박 간 시간 간격(h)"
]

vessel_info = vessel_df[vessel_cols].copy()


print("========== 모선항차 중복 검증 ==========")

print(
    "선박스케줄 중복 항차:",
    vessel_info["모선항차"].duplicated().sum()
)


container_berth_df = container_df.merge(
    vessel_info,
    left_on="매칭용_모선항차",
    right_on="모선항차",
    how="left",
    validate="many_to_one",
    suffixes=("", "_선석분석")
)

print("\n========== 컨테이너 병합 결과 ==========")

print("병합 전 컨테이너:", len(container_df))
print("병합 후 컨테이너:", len(container_berth_df))

print(
    "이전 선박 연결 컨테이너:",
    container_berth_df["이전_모선항차"].notna().sum()
)

target = "야드체류시간(h)"

metrics = [
    "이전 선박 작업시간(h)",
    "이전 선박 점유시간(h)",
    "선박 간 시간 간격(h)"
]


container_results = []

for metric in metrics:

    valid = (
        container_berth_df[[metric, target]]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    r = valid[metric].corr(
        valid[target],
        method="pearson"
    )

    container_results.append({
        "분석 지표": metric,
        "컨테이너 수": len(valid),
        "컨테이너 기준 r": round(r, 4)
    })

container_corr_df = pd.DataFrame(container_results)


ship_yard_df = (
    container_berth_df
    .groupby("매칭용_모선항차", as_index=False)
    .agg(
        평균_야드체류시간_h=("야드체류시간(h)", "mean"),
        컨테이너수=("컨테이너번호", "size")
    )
)

ship_analysis_df = ship_yard_df.merge(
    vessel_info,
    left_on="매칭용_모선항차",
    right_on="모선항차",
    how="left",
    validate="one_to_one"
)



ship_results = []

for metric in metrics:

    valid = (
        ship_analysis_df[
            [metric, "평균_야드체류시간_h"]
        ]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    r = valid[metric].corr(
        valid["평균_야드체류시간_h"],
        method="pearson"
    )

    ship_results.append({
        "분석 지표": metric,
        "선박 항차 수": len(valid),
        "선박 기준 r": round(r, 4)
    })

ship_corr_df = pd.DataFrame(ship_results)


comparison_df = container_corr_df.merge(
    ship_corr_df,
    on="분석 지표",
    how="outer"
)

print("\n========== 컨테이너 / 선박 기준 상관분석 비교 ==========")

display(comparison_df)


print("\n========== 선박별 야드 체류시간 기초 통계 ==========")

display(
    ship_analysis_df[
        ["평균_야드체류시간_h", "컨테이너수"]
    ].describe().round(1)
)


print("\n========== 선석별 연결 항차 수 ==========")

display(
    ship_analysis_df[
        ship_analysis_df["이전_모선항차"].notna()
    ]["선석"]
    .value_counts()
    .rename_axis("선석")
    .reset_index(name="분석 선박 항차 수")
)

========== 모선항차 중복 검증 ==========
선박스케줄 중복 항차: 0

========== 컨테이너 병합 결과 ==========
병합 전 컨테이너: 1490
병합 후 컨테이너: 1490
이전 선박 연결 컨테이너: 1490

========== 컨테이너 / 선박 기준 상관분석 비교 ==========


,분석 지표,컨테이너 수,컨테이너 기준 r,선박 항차 수,선박 기준 r
0,선박 간 시간 간격(h),1490,0.0263,378,0.0224
1,이전 선박 작업시간(h),1490,0.0368,378,-0.1553
2,이전 선박 점유시간(h),1490,0.0308,378,-0.1561



========== 선박별 야드 체류시간 기초 통계 ==========


,평균_야드체류시간_h,컨테이너수
count,378.0,378.0
mean,192.2,3.9
std,115.9,5.1
min,1.9,1.0
25%,113.3,1.0
50%,166.4,2.0
75%,257.4,4.0
max,777.1,38.0



========== 선석별 연결 항차 수 ==========


,선석,분석 선박 항차 수
0,2B,128
1,1B,118
2,3B,82
3,4B,50


##### 컨테이너 기준과 항차 기준 상관분석 결과

직전 선박 정보를 연결한 **컨테이너 1,490건(378개 적하 항차)**을 대상으로 분석 단위에 따른 상관계수를 비교했다.

| 지표 | 컨테이너 기준 r (1,490건) | 항차 기준 r (378항차) |
|:---|---:|---:|
| 이전 선박 작업시간 | 0.0368 | -0.1553 |
| 이전 선박 점유시간(대체지표) | 0.0308 | -0.1561 |
| 선박 간 시간 간격 | 0.0263 | 0.0224 |

컨테이너 기준에서는 모두 0에 가까웠고, 항차 기준에서는 직전 선박 작업시간과 입항~출항 경과시간이 약한 음의 상관관계를 보였다.

 **항차당 컨테이너 수는 1~38건**으로 달라 관측 단위별 가중 차이가 결과에 영향을 미쳤을 가능성이 있다. 
 
 따라서 직전 선박 운영시간만으로 후속 컨테이너 체류시간의 차이를 설명하기 어렵다.


### 6-4. 종합 결론 및 향후 시뮬레이션 방향

현재 적하 선박의 시간 지표(6-2)와 동일 선석의 직전 입항 선박 관련 지표(6-3)를 활용하여 컨테이너 야드 체류시간과의 선형 관계를 검토했다.

- 현재 선박 관련 5개 지표는 모두 **상관계수 절댓값 0.12 미만**이었다.
  
- 직전 선박 운영시간과 후속 선박 작업시작전 대기시간의 상관계수는 모두 **절댓값 0.04 미만**이었다.

- 직전 선박 관련 지표와 후속 컨테이너 야드 체류시간의 관계도 **컨테이너 기준에서는 거의 없었고**, 항차 기준에서는 일부 **약한 음의 상관관계**에 그쳤다.



**최종 결론:** 이 데이터에서 관찰한 선박 운영시간의 길이만으로 컨테이너별 야드 체류시간 차이를 설명하기는 어렵다.

 이는 선박 일정의 영향 자체가 없다는 뜻이 아니며, 상관관계 분석만으로 운영 병목이나 인과관계를 확정할 수 없다.





**향후 방향:** 다음 단계에서는 선박 입출항 일정과 컨테이너 야드 장치·온샤시 시각을 활용해 **운영 조건을 변경하는 시뮬레이션 시나리오**를 설계하고, 가정에 따른 야드 체류시간 변화 가능성을 비교하고자 한다. 

실제로 선박 일정 변경이 온샤시 시각에 미치는 영향은 이 자료만으로 확정할 수 없으므로, 시뮬레이션의 가정과 제약조건을 명시한 뒤 결과를 해석해야 한다.